In [10]:
import numpy as np
import pandas as pd
from obspy import read

xlsx = "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/data/processed/Shallow_processed_RESULTS.xlsx"
sheet = "best_7_bands_fixed_hold0"
MSEED_PATH = "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/notebooks/All_Shallow_Moonquakes.mseed"

FC = 5.0
BANDS = np.array([3., 4., 5., 6., 7., 8., 9.])
STARTTIME_TOL_S = 2.0
SCENARIOS = [dict(LOWER_TOL=0.75, MIN_POST=4, K_NEG=0, K_PRE_POS=0)]


def load_excel_long(xlsx, sheet, *, FC, BANDS):
    d = pd.read_excel(xlsx, sheet_name=sheet)
    need = ["starttime", "station", "fc_hz", "t0_dt_mean"]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(f"Missing columns in sheet '{sheet}': {missing}")

    d["station"] = d["station"].astype(str)
    d["fc_hz"] = pd.to_numeric(d["fc_hz"], errors="coerce").astype(float)
    d["starttime_dt"] = pd.to_datetime(d["starttime"], errors="coerce", utc=True)
    d["t0_dt_mean_dt"] = pd.to_datetime(d["t0_dt_mean"], errors="coerce", utc=True)

    if "distance" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["distance"], errors="coerce")
    elif "epi_deg" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["epi_deg"], errors="coerce")
    else:
        d["distance_deg"] = np.nan

    d = d[d["fc_hz"].isin(BANDS)].copy()
    d["event"] = d["starttime_dt"].astype(str) + "__" + d["station"]
    ref = (
        d[d["fc_hz"].eq(FC)][["event", "t0_dt_mean_dt"]]
        .rename(columns={"t0_dt_mean_dt": "t0_fc_dt"})
        .groupby("event", as_index=False)["t0_fc_dt"]
        .min()
    )
    d = d.merge(ref, on="event", how="left")
    d["dt_rel"] = (d["t0_dt_mean_dt"] - d["t0_fc_dt"]).dt.total_seconds()
    d = d[d["dt_rel"].notna() & d["starttime_dt"].notna() & d["t0_dt_mean_dt"].notna()].copy()
    return d[["event", "station", "starttime_dt", "fc_hz", "dt_rel", "distance_deg", "t0_dt_mean_dt"]].rename(columns={"fc_hz": "band"})


def build_event_band_matrix(df_long, *, BANDS):
    return (
        df_long.pivot_table(index="event", columns="band", values="dt_rel", aggfunc="first")
        .reindex(columns=BANDS)
        .sort_index()
    )


def select_events(*, dt_mat, FC, BANDS, MIN_POST, K_NEG, K_PRE_POS=0, LOWER_TOL=0.0):
    post_bands = [b for b in BANDS if b > FC]
    pre_bands = [b for b in BANDS if b < FC]
    keep = []
    for ev in dt_mat.index:
        dt = dt_mat.loc[ev]
        post_vals = dt[post_bands].dropna()
        if len(post_vals) < MIN_POST:
            keep.append(False)
            continue
        if int((post_vals <= LOWER_TOL).sum()) > K_NEG:
            keep.append(False)
            continue
        pre_vals = dt[pre_bands].dropna()
        if int((pre_vals > LOWER_TOL).sum()) > K_PRE_POS:
            keep.append(False)
            continue
        keep.append(True)
    return pd.Series(keep, index=dt_mat.index, name="keep")


def match_traces_to_excel_events(st, df_long, tol_s):
    by_sta = {sta: g.copy() for sta, g in df_long.groupby("station")}
    event_to_trace = {}
    for tr in st:
        sta = str(getattr(tr.stats, "station", "")).strip()
        if not sta or sta not in by_sta:
            continue
        tr_t0 = pd.Timestamp(tr.stats.starttime.datetime, tz="UTC")
        g = by_sta[sta]
        dt = (g["starttime_dt"] - tr_t0).dt.total_seconds().abs()
        j = dt.idxmin()
        if not np.isfinite(dt.loc[j]):
            continue
        if dt.loc[j] <= tol_s:
            ev = g.loc[j, "event"]
            if ev in event_to_trace:
                prev_tr, prev_diff = event_to_trace[ev]
                if dt.loc[j] < prev_diff:
                    event_to_trace[ev] = (tr, float(dt.loc[j]))
            else:
                event_to_trace[ev] = (tr, float(dt.loc[j]))
    return {ev: tr for ev, (tr, _) in event_to_trace.items()}


df_long = load_excel_long(xlsx, sheet, FC=FC, BANDS=BANDS)
dt_mat = build_event_band_matrix(df_long, BANDS=BANDS)
t0_mat = (
    df_long.pivot_table(index="event", columns="band", values="t0_dt_mean_dt", aggfunc="first")
    .reindex(columns=BANDS)
    .sort_index()
)
st = read(MSEED_PATH)
event_to_trace = match_traces_to_excel_events(st, df_long, tol_s=STARTTIME_TOL_S)

selected_rows = []
for cfg in SCENARIOS:
    keep_mask = select_events(
        dt_mat=dt_mat,
        FC=FC,
        BANDS=BANDS,
        MIN_POST=int(cfg["MIN_POST"]),
        K_NEG=int(cfg["K_NEG"]),
        K_PRE_POS=int(cfg["K_PRE_POS"]),
        LOWER_TOL=float(cfg["LOWER_TOL"]),
    )
    kept_events = [ev for ev in keep_mask.index[keep_mask].tolist() if ev in event_to_trace]
    for ev in kept_events:
        station = ev.split("__", 1)[-1]
        t0_fc = df_long.loc[(df_long["event"] == ev) & (df_long["band"] == FC), "t0_dt_mean_dt"]
        distance_deg = df_long.loc[df_long["event"] == ev, "distance_deg"].dropna()
        selected_rows.append({
            "event": ev,
            "station": station,
            "time_utc": t0_fc.iloc[0] if not t0_fc.empty else pd.NaT,
            "epi_deg": float(distance_deg.iloc[0]) if not distance_deg.empty else np.nan,
        })

selected_events_df = (
    pd.DataFrame(selected_rows)
    .drop_duplicates(subset=["event"])
    .sort_values(["epi_deg", "time_utc"], na_position="last")
    .reset_index(drop=True)
)
print(f"Matched {len(event_to_trace)} Excel events to MiniSEED traces.")
print(selected_events_df.to_string(index=False))

Matched 43 Excel events to MiniSEED traces.
                                event station                         time_utc    epi_deg
1976-03-06 10:03:00.009000+00:00__S14     S14 1976-03-06 10:15:48.794000+00:00  56.580047
1975-01-03 01:32:00.013000+00:00__S15     S15 1975-01-03 01:47:03.344000+00:00  83.980853
1973-03-13 07:47:00.014000+00:00__S14     S14 1973-03-13 08:01:35.960000+00:00  88.890000
1974-07-11 00:37:00.006000+00:00__S14     S14 1974-07-11 00:52:15.850000+00:00 100.704577


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.signal import butter, sosfiltfilt, hilbert, get_window

MOON_RADIUS_KM = 1737.4
RESULTS_DIR = os.path.join(os.getcwd(), "results", "p_envelope_peak_delay")
EVENT_FIG_DIR = os.path.join(RESULTS_DIR, "event_figures")
os.makedirs(EVENT_FIG_DIR, exist_ok=True)

HALF_BW = 0.5
BP_ORDER = 4
SMOOTH_WINDOW_S = 5.0
NOISE_START = -150.0
NOISE_END = -30.0
ONSET_SEARCH_START = -20.0
ONSET_SEARCH_END = 120.0
FIT_START_REL_T0_S = 0.0
P_WINDOW_S = 50.0
ONSET_SIGMA = 3.0
MIN_NOISE_N = 40
MIN_ONSET_SEC = 5.0
MAX_PLOT_TIME = 180.0
RESULTS_CSV = os.path.join(RESULTS_DIR, "p_envelope_peak_delay_measurements.csv")


def compute_bandpass_limits(center_hz, half_width_hz, sampling_hz):
    low_hz = max(center_hz - half_width_hz, 0.001)
    high_hz = min(center_hz + half_width_hz, 0.99 * sampling_hz / 2)
    return low_hz, high_hz


def build_rms_envelope(signal, sampling_hz, low_hz, high_hz, smooth_window_s, filter_order):
    sos = butter(filter_order, [low_hz / (sampling_hz / 2), high_hz / (sampling_hz / 2)], btype="bandpass", output="sos")
    bandpassed = sosfiltfilt(sos, signal)
    analytic_env = np.abs(hilbert(bandpassed))
    nwin = int(round(smooth_window_s * sampling_hz)) | 1
    window = get_window("hann", nwin)
    window = window / window.sum()
    return np.sqrt(np.convolve(analytic_env ** 2, window, mode="same"))


def estimate_noise_stats(time_s, envelope, noise_start, noise_end, min_noise_n):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (envelope > 0) & (time_s >= noise_start) & (time_s <= noise_end)
    if mask.sum() < min_noise_n:
        raise RuntimeError("Not enough points in pre-event noise window")
    noise_vals = envelope[mask]
    return float(np.median(noise_vals)), float(np.std(noise_vals, ddof=0))


def detect_onset(time_s, envelope, *, threshold, onset_search_start, onset_search_end, min_onset_sec, sampling_hz):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (time_s >= onset_search_start) & (time_s <= onset_search_end)
    if mask.sum() == 0:
        raise RuntimeError("No samples in onset search window")
    t_sel = time_s[mask]
    e_sel = envelope[mask]
    above = e_sel >= threshold
    n_consecutive = max(1, int(round(min_onset_sec * sampling_hz)))
    run = 0
    for idx, is_above in enumerate(above):
        run = run + 1 if is_above else 0
        if run >= n_consecutive:
            return float(t_sel[idx - n_consecutive + 1])
    raise RuntimeError("No onset crossing found")


def find_peak_in_window(time_s, envelope, onset_s, p_window_s):
    p_window_end_s = onset_s + p_window_s
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (time_s >= onset_s) & (time_s <= p_window_end_s)
    if mask.sum() == 0:
        raise RuntimeError("No samples in P window")
    t_sel = time_s[mask]
    e_sel = envelope[mask]
    idx = int(np.argmax(e_sel))
    return float(t_sel[idx]), float(e_sel[idx]), float(p_window_end_s)


def first_crossing_time(time_s, envelope, onset_s, end_s, target_amp):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (time_s >= onset_s) & (time_s <= end_s)
    if mask.sum() == 0:
        return np.nan
    t_sel = time_s[mask]
    e_sel = envelope[mask]
    idx = np.flatnonzero(e_sel >= target_amp)
    return float(t_sel[idx[0]]) if idx.size else np.nan


def fit_saturating_rise(time_s, envelope, fit_start_s, end_s, baseline):
    mask = np.isfinite(time_s) & np.isfinite(envelope) & (envelope > 0) & (time_s >= fit_start_s) & (time_s <= end_s)
    if mask.sum() < 3:
        raise RuntimeError("Not enough points in P rise window")
    t_fit = time_s[mask]
    y_fit = envelope[mask]
    x_fit = t_fit - fit_start_s
    baseline = max(float(baseline), 1e-30)

    def model(params):
        amp, tau = params
        amp = max(float(amp), 1e-30)
        tau = max(float(tau), 1e-6)
        return baseline + amp * (1.0 - np.exp(-x_fit / tau))

    amp0 = max(float(np.max(y_fit) - baseline), baseline)
    tau0 = max((end_s - fit_start_s) / 4.0, 1.0)
    result = least_squares(
        lambda p: np.log(np.clip(model(p), 1e-30, None)) - np.log(np.clip(y_fit, 1e-30, None)),
        x0=np.array([amp0, tau0], float),
        bounds=(np.array([1e-30, 1e-6]), np.array([np.inf, max(1.0, 10.0 * (end_s - fit_start_s))])),
        method="trf",
        max_nfev=2000,
    )
    if not result.success:
        raise RuntimeError(result.message)
    amp, tau = result.x
    env_model = model(result.x)
    rmse_log = float(np.sqrt(np.mean((np.log(np.clip(env_model, 1e-30, None)) - np.log(np.clip(y_fit, 1e-30, None))) ** 2)))
    return float(amp), float(tau), rmse_log, t_fit, env_model


def plot_event_measurements(event_id, station, epicentral_deg, band_rows, output_dir):
    nrows = len(band_rows)
    fig, axes = plt.subplots(nrows, 2, figsize=(14, max(3.0 * nrows, 4.5)), sharex=True)
    if nrows == 1:
        axes = np.array([axes])
    for ax_row, row in zip(axes, band_rows):
        ax_lin, ax_log = ax_row
        for ax in ax_row:
            ax.axvspan(row["fit_start_plot_s"], row["p_window_end_plot_s"], color="tab:blue", alpha=0.08)
            ax.axvline(0.0, color="0.5", ls=":")
            ax.axvline(row["fit_start_plot_s"], color="tab:orange", ls="--")
            ax.axvline(row["onset_plot_s"], color="tab:blue", ls="--")
            ax.axvline(row["peak_plot_s"], color="tab:green", ls="--")
            ax.axvline(row["p_window_end_plot_s"], color="tab:purple", ls=":")
        ax_lin.plot(row["time_plot_s"], row["envelope"], color="k", lw=1)
        ax_lin.axhline(row["threshold"], color="tab:red", ls=":")
        ax_lin.plot(row["t_model_plot_s"], row["env_model"], color="crimson", lw=2)
        ax_lin.set_xlim(-60, MAX_PLOT_TIME)
        ax_lin.set_ylabel(f"{row['fc_Hz']:.1f} Hz")
        ax_lin.set_title(f"Linear | dt0={row['band_t0_rel_s']:.1f}s | onset={row['onset_s']:.1f}s | peak={row['peak_s']:.1f}s")
        ax_lin.grid(True, alpha=0.3)

        ax_log.semilogy(row["time_plot_s"], row["envelope"], color="k", lw=1)
        ax_log.axhline(row["threshold"], color="tab:red", ls=":")
        ax_log.semilogy(row["t_model_plot_s"], row["env_model"], color="crimson", lw=2)
        ax_log.set_xlim(-60, MAX_PLOT_TIME)
        ax_log.set_title(f"Semilogy | dtp={row['delta_tp_s']:.1f}s | rise10-90={row['rise_10_90_s']:.1f}s | tau={row['tau_rise_s']:.1f}s | rmse={row['rmse_log']:.3f}")
        ax_log.grid(True, which="both", alpha=0.3)
    axes[-1, 0].set_xlabel(f"Time relative to {FC:.1f} Hz t0 (s)")
    axes[-1, 1].set_xlabel(f"Time relative to {FC:.1f} Hz t0 (s)")
    fig.suptitle(f"P-envelope peak-delay observables | {event_id} | station={station} | distance={epicentral_deg:.2f} deg")
    fig.tight_layout()
    safe_event = "".join(ch if ch.isalnum() else "_" for ch in event_id)
    out_path = os.path.join(output_dir, f"{safe_event}_p_envelope_peak_delay.png")
    fig.savefig(out_path, dpi=200)
    plt.show()
    return out_path


measurements = []
failed_rows = []

for event_idx, row in selected_events_df.iterrows():
    ev = row["event"]
    tr = event_to_trace.get(ev)
    if tr is None:
        failed_rows.append({"event": ev, "fc_Hz": np.nan, "reason": "Missing MiniSEED trace"})
        continue

    tr_start_utc = pd.Timestamp(tr.stats.starttime.datetime, tz="UTC")
    fs = float(tr.stats.sampling_rate)
    signal = tr.data.astype(float)
    t_raw = np.arange(signal.size) / fs
    band_rows = []
    epi_km = MOON_RADIUS_KM * np.deg2rad(float(row["epi_deg"]))

    print(f"Processing event {event_idx + 1}/{len(selected_events_df)}: {ev}")

    for fc_band in BANDS:
        try:
            t0_utc = t0_mat.loc[ev, fc_band]
        except KeyError:
            t0_utc = pd.NaT
        if pd.isna(t0_utc):
            failed_rows.append({"event": ev, "fc_Hz": float(fc_band), "reason": "Missing t0_dt_mean"})
            continue

        t0_utc = pd.Timestamp(t0_utc)
        t0_utc = t0_utc.tz_localize("UTC") if t0_utc.tzinfo is None else t0_utc.tz_convert("UTC")
        band_t0_rel_s = float(dt_mat.loc[ev, fc_band]) if pd.notna(dt_mat.loc[ev, fc_band]) else 0.0
        t0_offset_s = (t0_utc - tr_start_utc).total_seconds()
        time_s = t_raw - t0_offset_s

        fl_hz, fu_hz = compute_bandpass_limits(fc_band, HALF_BW, fs)
        envelope = build_rms_envelope(signal, fs, fl_hz, fu_hz, SMOOTH_WINDOW_S, BP_ORDER)
        if envelope.size != time_s.size:
            i0 = max(0, (time_s.size - envelope.size) // 2)
            time_s = time_s[i0:i0 + envelope.size]

        try:
            noise_median, noise_std = estimate_noise_stats(time_s, envelope, NOISE_START, NOISE_END, MIN_NOISE_N)
            threshold = noise_median + ONSET_SIGMA * noise_std
            onset_s = detect_onset(
                time_s,
                envelope,
                threshold=threshold,
                onset_search_start=ONSET_SEARCH_START,
                onset_search_end=ONSET_SEARCH_END,
                min_onset_sec=MIN_ONSET_SEC,
                sampling_hz=fs,
            )
            peak_s, peak_amp, p_window_end_s = find_peak_in_window(time_s, envelope, onset_s, P_WINDOW_S)
            t10_s = first_crossing_time(time_s, envelope, onset_s, p_window_end_s, 0.10 * peak_amp)
            t50_s = first_crossing_time(time_s, envelope, onset_s, p_window_end_s, 0.50 * peak_amp)
            t90_s = first_crossing_time(time_s, envelope, onset_s, p_window_end_s, 0.90 * peak_amp)
            rise_10_90_s = float(t90_s - t10_s) if np.isfinite(t10_s) and np.isfinite(t90_s) and t90_s >= t10_s else np.nan
            fit_start_s = FIT_START_REL_T0_S
            amp_rise, tau_rise_s, rmse_log, t_model, env_model = fit_saturating_rise(time_s, envelope, fit_start_s, p_window_end_s, noise_median)
        except Exception as exc:
            failed_rows.append({"event": ev, "fc_Hz": float(fc_band), "reason": str(exc)})
            continue

        delta_tp_s = float(peak_s - onset_s)
        time_plot_s = time_s + band_t0_rel_s
        t_model_plot_s = t_model + band_t0_rel_s
        band_rows.append({
            "fc_Hz": float(fc_band),
            "time_s": time_s.copy(),
            "time_plot_s": time_plot_s.copy(),
            "envelope": envelope.copy(),
            "threshold": threshold,
            "band_t0_rel_s": band_t0_rel_s,
            "fit_start_s": fit_start_s,
            "fit_start_plot_s": fit_start_s + band_t0_rel_s,
            "onset_s": onset_s,
            "onset_plot_s": onset_s + band_t0_rel_s,
            "peak_s": peak_s,
            "peak_plot_s": peak_s + band_t0_rel_s,
            "p_window_end_s": p_window_end_s,
            "p_window_end_plot_s": p_window_end_s + band_t0_rel_s,
            "delta_tp_s": delta_tp_s,
            "rise_10_90_s": rise_10_90_s,
            "tau_rise_s": tau_rise_s,
            "rmse_log": rmse_log,
            "t_model": t_model,
            "t_model_plot_s": t_model_plot_s,
            "env_model": env_model,
        })
        measurements.append({
            "event": ev,
            "station": row["station"],
            "epi_deg": float(row["epi_deg"]),
            "epi_km": float(epi_km),
            "event_time_utc": row["time_utc"],
            "fc_Hz": float(fc_band),
            "band_t0_rel_s": band_t0_rel_s,
            "noise_median": noise_median,
            "noise_std": noise_std,
            "threshold": threshold,
            "fit_start_s": fit_start_s,
            "onset_s": onset_s,
            "peak_s": peak_s,
            "p_window_end_s": p_window_end_s,
            "delta_tp_s": delta_tp_s,
            "peak_amp": peak_amp,
            "t10_s": t10_s,
            "t50_s": t50_s,
            "t90_s": t90_s,
            "rise_10_90_s": rise_10_90_s,
            "amp_rise": amp_rise,
            "tau_rise_s": tau_rise_s,
            "rmse_log": rmse_log,
        })

    if band_rows:
        plot_event_measurements(ev, row["station"], row["epi_deg"], band_rows, EVENT_FIG_DIR)

measurements_df = pd.DataFrame(measurements)
failed_measurements_df = pd.DataFrame(failed_rows)
if measurements_df.empty:
    print("No successful P-envelope measurements were produced.")
else:
    measurements_df = measurements_df.sort_values(["event_time_utc", "fc_Hz"]).reset_index(drop=True)
    measurements_df.to_csv(RESULTS_CSV, index=False)
    summary_cols = ["event", "station", "epi_deg", "fc_Hz", "band_t0_rel_s", "fit_start_s", "onset_s", "peak_s", "delta_tp_s", "rise_10_90_s", "rmse_log"]
    print(measurements_df[summary_cols].to_string(index=False))
    print()
    print(f"Saved measurements to: {RESULTS_CSV}")
if not failed_measurements_df.empty:
    print()
    print("Failed measurements:")
    print(failed_measurements_df.to_string(index=False))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

RESULTS_DIR = os.path.join(os.getcwd(), "results", "p_envelope_peak_delay")
SUMMARY_CSV = os.path.join(RESULTS_DIR, "p_envelope_peak_delay_summary_by_band.csv")
FIT_CSV = os.path.join(RESULTS_DIR, "p_envelope_peak_delay_global_fits.csv")
SCATTERING_CSV = os.path.join(RESULTS_DIR, "p_envelope_scattering_parameters.csv")
VP_KM_S = 4.0
D0_KM = 1000.0
F0_HZ = 1.0

if measurements_df.empty:
    raise RuntimeError("Run the P-envelope processing cell successfully before the summary analysis.")

print("Summary by event and band:")
print(measurements_df[["event", "station", "epi_deg", "epi_km", "fc_Hz", "band_t0_rel_s", "onset_s", "peak_s", "delta_tp_s", "rise_10_90_s", "rmse_log"]].to_string(index=False))

print()
print("Summary by frequency band:")
summary_by_band = (
    measurements_df.groupby("fc_Hz")[["delta_tp_s", "rise_10_90_s"]]
    .agg(["count", "mean", "std", "median", "min", "max"])
)
summary_by_band.to_csv(SUMMARY_CSV)
print(summary_by_band.to_string())


def fit_power_law_surface(df, value_col):
    valid = df[(df[value_col] > 0) & (df["epi_km"] > 0) & (df["fc_Hz"] > 0)].copy()
    if len(valid) < 4:
        raise RuntimeError(f"Not enough valid points to fit {value_col}")
    y = np.log(valid[value_col].to_numpy(float))
    x_d = np.log(valid["epi_km"].to_numpy(float))
    x_f = np.log(valid["fc_Hz"].to_numpy(float))

    def residuals(params):
        log_a, b, c = params
        return (log_a + b * x_d + c * x_f) - y

    x0 = np.array([0.0, 1.0, -1.0], float)
    result = least_squares(residuals, x0, method="trf")
    if not result.success:
        raise RuntimeError(result.message)
    log_a, b, c = result.x
    a = float(np.exp(log_a))
    valid[f"{value_col}_pred"] = a * valid["epi_km"] ** b * valid["fc_Hz"] ** c
    rmse_log = float(np.sqrt(np.mean((np.log(valid[f"{value_col}_pred"]) - np.log(valid[value_col])) ** 2)))
    return {"value": value_col, "A": a, "b_distance": float(b), "c_frequency": float(c), "rmse_log": rmse_log}, valid

fit_rows = []
fitted_tables = {}
for value_col in ["delta_tp_s", "rise_10_90_s"]:
    try:
        fit_row, fitted_df = fit_power_law_surface(measurements_df, value_col)
        fit_rows.append(fit_row)
        fitted_tables[value_col] = fitted_df
        print()
        print(f"Global fit for {value_col}: {fit_row['A']:.4e} * D_km^{fit_row['b_distance']:.3f} * f_Hz^{fit_row['c_frequency']:.3f}")
        print(f"Log-space RMSE = {fit_row['rmse_log']:.3f}")
    except Exception as exc:
        print()
        print(f"Could not fit {value_col}: {exc}")

fit_results_df = pd.DataFrame(fit_rows)
if not fit_results_df.empty:
    fit_results_df.to_csv(FIT_CSV, index=False)

for value_col, ylabel in [("delta_tp_s", "Peak delay (s)"), ("rise_10_90_s", "Rise 10-90% (s)")]:
    if value_col not in fitted_tables:
        continue
    fitted_df = fitted_tables[value_col]
    print()
    print(f"Per-band plots for {value_col}:")
    for fc_band in sorted(fitted_df["fc_Hz"].unique()):
        band_df = fitted_df.loc[fitted_df["fc_Hz"] == fc_band].sort_values("epi_km")
        d_plot = np.linspace(band_df["epi_km"].min(), band_df["epi_km"].max(), 100)
        fit_row = fit_results_df.loc[fit_results_df["value"] == value_col].iloc[0]
        y_plot = fit_row["A"] * d_plot ** fit_row["b_distance"] * fc_band ** fit_row["c_frequency"]
        fig, ax = plt.subplots(figsize=(6.5, 4.0))
        ax.scatter(band_df["epi_km"], band_df[value_col], color="k", label="Observed")
        ax.plot(d_plot, y_plot, color="crimson", lw=2, label="Global fit")
        ax.set_xlabel("Epicentral distance (km)")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{value_col} | {fc_band:.1f} Hz")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        plt.show()

scattering_rows = []
delta_fit = fit_results_df.loc[fit_results_df["value"] == "delta_tp_s"]
if delta_fit.empty:
    print()
    print("Could not derive scattering parameters because the delta_tp_s fit is unavailable.")
else:
    delta_fit = delta_fit.iloc[0]
    A = float(delta_fit["A"])
    b = float(delta_fit["b_distance"])
    c = float(delta_fit["c_frequency"])

    print()
    print("Scattering interpretation from delta_tp_s:")
    print(f"Assumed vp = {VP_KM_S:.2f} km/s")
    print("Using diffusion scaling: delta_tp ~ L^2 / d(f), with L approximated by epicentral distance D.")
    print(f"Empirical fit: delta_tp = {A:.4e} * D_km^{b:.3f} * f_Hz^{c:.3f}")

    # For an ideal diffusion interpretation, b should be near 2 and c should be negative.
    # We map the fitted scaling to an effective diffusivity using d_eff = D^2 / delta_tp.
    # This gives d_eff(D, f) = (1/A) * D^(2-b) * f^(-c).
    freq_grid = sorted(measurements_df["fc_Hz"].unique())
    for fc_band in freq_grid:
        d_eff_ref = (D0_KM ** 2) / (A * D0_KM ** b * fc_band ** c)
        mu_eff_ref = 3.0 * d_eff_ref / VP_KM_S
        qsc_eff_ref = 2.0 * np.pi * fc_band * mu_eff_ref / VP_KM_S
        scattering_rows.append({
            "fc_Hz": float(fc_band),
            "D_ref_km": D0_KM,
            "vp_km_s": VP_KM_S,
            "delta_tp_ref_s": A * D0_KM ** b * fc_band ** c,
            "d_eff_km2_s": d_eff_ref,
            "mu_eff_km": mu_eff_ref,
            "Qsc_eff": qsc_eff_ref,
            "distance_exponent_for_d": float(2.0 - b),
            "frequency_exponent_for_d": float(-c),
        })

    scattering_df = pd.DataFrame(scattering_rows).sort_values("fc_Hz")
    scattering_df.to_csv(SCATTERING_CSV, index=False)
    print()
    print("Effective scattering parameters at D_ref = 1000 km:")
    print(scattering_df.to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(scattering_df["fc_Hz"], scattering_df["d_eff_km2_s"], "o-", color="k")
    axes[0].set_xlabel("Frequency (Hz)")
    axes[0].set_ylabel("d_eff (km^2/s)")
    axes[0].set_title("Effective diffusivity")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(scattering_df["fc_Hz"], scattering_df["mu_eff_km"], "o-", color="k")
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("mu_eff (km)")
    axes[1].set_title("Effective mean free path")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(scattering_df["fc_Hz"], scattering_df["Qsc_eff"], "o-", color="k")
    axes[2].set_xlabel("Frequency (Hz)")
    axes[2].set_ylabel("Qsc_eff")
    axes[2].set_title("Effective scattering Q")
    axes[2].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

print()
print(f"Saved band summary to: {SUMMARY_CSV}")
if not fit_results_df.empty:
    print(f"Saved global fits to: {FIT_CSV}")
if os.path.exists(SCATTERING_CSV):
    print(f"Saved scattering parameters to: {SCATTERING_CSV}")